# Time-Series Forecasting Benchmark

## ETS vs LightGBM vs Chronos-2

This notebook compares three forecasting paradigms across two public retail-demand datasets:

- **ETS** - statistical forecasting
- **LightGBM** - global machine-learning forecasting
- **Chronos-2** - pretrained time-series foundation model

The common protocol uses 500 daily series per dataset, a 28-day validation window, and a final 28-day test horizon. Models are compared on **sMAPE, WAPE, RMSE, uncertainty calibration, and computational cost**.

A separate M5 experiment tests whether adding price, event, and SNAP covariates improves LightGBM.


## 1. Setup

A GPU is recommended for Chronos-2 inference. ETS and the default LightGBM experiments run on CPU.


In [ ]:
%pip install -q lightgbm chronos-forecasting

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from lightgbm import LGBMRegressor
from chronos import Chronos2Pipeline

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 2. Data files

The datasets are intentionally not stored in this repository.

Download the Kaggle **Store Item Demand Forecasting Challenge** and **M5 Forecasting - Accuracy** datasets and place these extracted files in a local `data/` directory:

- `train.csv`
- `sales_train_validation.csv`
- `calendar.csv`
- `sell_prices.csv`

In Google Colab, upload them to `/content/data/`.


In [ ]:
DATA_DIR = Path("/content/data") if Path("/content").exists() else Path("data")

STORE_ITEM_PATH = DATA_DIR / "train.csv"
M5_SALES_PATH = DATA_DIR / "sales_train_validation.csv"
M5_CALENDAR_PATH = DATA_DIR / "calendar.csv"
M5_PRICES_PATH = DATA_DIR / "sell_prices.csv"

required = [STORE_ITEM_PATH, M5_SALES_PATH, M5_CALENDAR_PATH, M5_PRICES_PATH]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing dataset files:\n" + "\n".join(missing)
    )


## 3. Shared evaluation utilities

Forecasting splits are chronological to avoid future-to-past leakage. Lower values are better for sMAPE, WAPE, and RMSE.


In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.abs(y_true) + np.abs(y_pred)
    values = np.where(
        denominator == 0,
        0,
        2 * np.abs(y_pred - y_true) / denominator
    )
    return 100 * np.mean(values)


def wape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100 * np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def chronological_split(df, date_col="date", horizon=28):
    max_date = df[date_col].max()
    test_start = max_date - pd.Timedelta(days=horizon - 1)
    val_start = test_start - pd.Timedelta(days=horizon)

    train = df[df[date_col] < val_start].copy()
    val = df[
        (df[date_col] >= val_start) &
        (df[date_col] < test_start)
    ].copy()
    test = df[df[date_col] >= test_start].copy()

    return train, val, test


## 4. Generic ETS benchmark

ETS is fitted separately to every series with weekly seasonality. The same two candidate configurations are considered on both datasets:

- additive trend + additive seasonality
- damped additive trend + additive seasonality


In [ ]:
ETS_CONFIGS = {
    "Additive Trend": {
        "trend": "add",
        "seasonal": "add",
        "damped_trend": False
    },
    "Damped Trend": {
        "trend": "add",
        "seasonal": "add",
        "damped_trend": True
    }
}


def run_ets(train_data, future_data, series_cols, config):
    predictions = []
    failures = []
    fit_time = 0.0
    inference_time = 0.0

    grouper = series_cols[0] if len(series_cols) == 1 else series_cols

    future_groups = {
        key: group.sort_values("date")
        for key, group in future_data.groupby(grouper)
    }

    for key, train_series in train_data.groupby(grouper):
        train_series = train_series.sort_values("date")
        future_series = future_groups[key]
        y_train = train_series["sales"].values

        try:
            start = time.perf_counter()
            model = ExponentialSmoothing(
                y_train,
                seasonal_periods=7,
                initialization_method="estimated",
                **config
            )
            fitted = model.fit(optimized=True)
            fit_time += time.perf_counter() - start

            start = time.perf_counter()
            forecast = np.maximum(
                fitted.forecast(len(future_series)),
                0
            )
            inference_time += time.perf_counter() - start

            key_tuple = key if isinstance(key, tuple) else (key,)

            for (_, row), pred in zip(future_series.iterrows(), forecast):
                record = {
                    "date": row["date"],
                    "actual": row["sales"],
                    "forecast": float(pred)
                }
                for col, value in zip(series_cols, key_tuple):
                    record[col] = value
                predictions.append(record)

        except Exception as exc:
            failures.append((key, str(exc)))

    return pd.DataFrame(predictions), fit_time, inference_time, failures


def ets_validation_benchmark(train_df, val_df, series_cols):
    rows = []
    forecasts = {}

    for name, config in ETS_CONFIGS.items():
        print("ETS:", name)

        preds, fit_time, inference_time, failures = run_ets(
            train_df, val_df, series_cols, config
        )
        forecasts[name] = preds

        rows.append({
            "configuration": name,
            "sMAPE": smape(preds["actual"], preds["forecast"]),
            "WAPE": wape(preds["actual"], preds["forecast"]),
            "RMSE": rmse(preds["actual"], preds["forecast"]),
            "fit_time_seconds": fit_time,
            "inference_time_seconds": inference_time,
            "failures": len(failures)
        })

    results = pd.DataFrame(rows)
    best_name = results.loc[results["sMAPE"].idxmin(), "configuration"]
    return results, best_name, forecasts[best_name]


## 5. Generic global LightGBM benchmark

One global LightGBM model is trained across all series using identifiers, calendar features, sales lags, and rolling averages. Forecasts are generated recursively for the 28-day horizon.


In [ ]:
LGBM_CONFIGS = {
    "Config A": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 31
    },
    "Config B": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 31
    }
}


def make_lgbm_features(df, series_col, static_cols):
    data = df.sort_values([series_col, "date"]).copy()

    data["day_of_week"] = data["date"].dt.dayofweek
    data["month"] = data["date"].dt.month
    data["day_of_year"] = data["date"].dt.dayofyear

    for lag in [1, 7, 14, 28]:
        data[f"lag_{lag}"] = (
            data.groupby(series_col)["sales"].shift(lag)
        )

    data["rolling_mean_7"] = (
        data.groupby(series_col)["sales"]
        .transform(lambda x: x.shift(1).rolling(7).mean())
    )
    data["rolling_mean_28"] = (
        data.groupby(series_col)["sales"]
        .transform(lambda x: x.shift(1).rolling(28).mean())
    )

    feature_cols = static_cols + [
        "day_of_week", "month", "day_of_year",
        "lag_1", "lag_7", "lag_14", "lag_28",
        "rolling_mean_7", "rolling_mean_28"
    ]

    return data, feature_cols


def fit_lgbm(train_df, series_col, static_cols, config):
    features, feature_cols = make_lgbm_features(
        train_df, series_col, static_cols
    )
    features = features.dropna(subset=feature_cols)

    model = LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1,
        **config
    )

    start = time.perf_counter()
    model.fit(features[feature_cols], features["sales"])
    fit_time = time.perf_counter() - start

    return model, feature_cols, fit_time


def recursive_lgbm_forecast(
    model,
    history_df,
    future_df,
    series_col,
    static_cols,
    feature_cols
):
    history = {
        key: group.sort_values("date")["sales"].tolist()
        for key, group in history_df.groupby(series_col)
    }

    static_values = (
        history_df.sort_values("date")
        .groupby(series_col)[static_cols]
        .first()
        .to_dict("index")
    )

    rows_out = []
    start = time.perf_counter()

    for date in sorted(future_df["date"].unique()):
        date = pd.Timestamp(date)
        rows = []

        for key, hist in history.items():
            row = {
                series_col: key,
                "day_of_week": date.dayofweek,
                "month": date.month,
                "day_of_year": date.dayofyear,
                "lag_1": hist[-1],
                "lag_7": hist[-7],
                "lag_14": hist[-14],
                "lag_28": hist[-28],
                "rolling_mean_7": np.mean(hist[-7:]),
                "rolling_mean_28": np.mean(hist[-28:])
            }
            row.update(static_values[key])
            rows.append(row)

        X_future = pd.DataFrame(rows)[feature_cols]
        preds = np.maximum(model.predict(X_future), 0)

        for row, pred in zip(rows, preds):
            key = row[series_col]
            history[key].append(float(pred))
            rows_out.append({
                series_col: key,
                "date": date,
                "forecast": float(pred)
            })

    inference_time = time.perf_counter() - start

    actual = future_df[
        [series_col, "date", "sales"]
    ].rename(columns={"sales": "actual"})

    preds = pd.DataFrame(rows_out).merge(
        actual,
        on=[series_col, "date"],
        how="left"
    )

    return preds, inference_time


def lgbm_validation_benchmark(
    train_df,
    val_df,
    series_col,
    static_cols,
    configs
):
    rows = []
    forecasts = {}

    for name, config in configs.items():
        print("LightGBM:", name)

        model, feature_cols, fit_time = fit_lgbm(
            train_df, series_col, static_cols, config
        )
        preds, inference_time = recursive_lgbm_forecast(
            model,
            train_df,
            val_df,
            series_col,
            static_cols,
            feature_cols
        )
        forecasts[name] = preds

        rows.append({
            "configuration": name,
            "sMAPE": smape(preds["actual"], preds["forecast"]),
            "WAPE": wape(preds["actual"], preds["forecast"]),
            "RMSE": rmse(preds["actual"], preds["forecast"]),
            "fit_time_seconds": fit_time,
            "inference_time_seconds": inference_time
        })

    results = pd.DataFrame(rows)
    best_name = results.loc[results["sMAPE"].idxmin(), "configuration"]
    return results, best_name, forecasts[best_name]


## 6. Chronos-2 helper

Chronos-2 is used zero-shot. Its 0.1 and 0.9 quantiles form the native 80% prediction interval.


In [ ]:
chronos_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=DEVICE
)


def run_chronos(
    history_df,
    test_df,
    series_col,
    prediction_length=28,
    batch_size=64
):
    contexts = []
    keys = []

    for key, group in history_df.groupby(series_col):
        group = group.sort_values("date")
        contexts.append(
            torch.tensor(group["sales"].values, dtype=torch.float32)
        )
        keys.append(key)

    start = time.perf_counter()

    quantiles, means = chronos_pipeline.predict_quantiles(
        contexts,
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
        batch_size=batch_size
    )

    inference_time = time.perf_counter() - start
    rows = []

    for key, q, mean in zip(keys, quantiles, means):
        q = q.squeeze(0).cpu().numpy()
        mean = mean.squeeze(0).cpu().numpy()

        actual = (
            test_df[test_df[series_col] == key]
            .sort_values("date")
        )

        for date, y, pred, lo, hi in zip(
            actual["date"],
            actual["sales"],
            mean,
            q[:, 0],
            q[:, 2]
        ):
            rows.append({
                series_col: key,
                "date": date,
                "actual": y,
                "forecast": max(float(pred), 0),
                "lower_80": max(float(lo), 0),
                "upper_80": max(float(hi), 0)
            })

    return pd.DataFrame(rows), inference_time


# 7. Dataset 1 - Store Item Demand

The Store Item dataset contains 10 stores × 50 items = **500 daily series** and 913,000 rows. It is used as the relatively simple public benchmark.


In [ ]:
store_item = pd.read_csv(STORE_ITEM_PATH)
store_item["date"] = pd.to_datetime(store_item["date"])
store_item = store_item.sort_values(
    ["store", "item", "date"]
).reset_index(drop=True)

store_item["series_key"] = (
    store_item["store"].astype(str) + "_" +
    store_item["item"].astype(str)
)

print("Shape:", store_item.shape)
print("Series:", store_item["series_key"].nunique())
print("Date range:", store_item["date"].min(), "to", store_item["date"].max())
print("Missing values:", store_item.isna().sum().sum())

example = store_item[
    (store_item["store"] == 1) &
    (store_item["item"] == 1)
]

plt.figure(figsize=(12, 4))
plt.plot(example["date"], example["sales"])
plt.title("Store 1 - Item 1 daily sales")
plt.xlabel("Date")
plt.ylabel("Units sold")
plt.show()

weekly = (
    store_item.assign(day=store_item["date"].dt.dayofweek)
    .groupby("day")["sales"].mean()
)

plt.figure(figsize=(8, 4))
plt.bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], weekly.values)
plt.title("Average sales by day of week")
plt.ylabel("Average units sold")
plt.show()


In [ ]:
d1_train, d1_val, d1_test = chronological_split(store_item)

print("Train:", d1_train["date"].min(), "to", d1_train["date"].max())
print("Validation:", d1_val["date"].min(), "to", d1_val["date"].max())
print("Test:", d1_test["date"].min(), "to", d1_test["date"].max())

print("Validation length per series:")
print(d1_val.groupby("series_key").size().value_counts())


## 7.1 Dataset 1 - ETS


In [ ]:
d1_ets_validation, d1_best_ets_name, d1_ets_val_preds = (
    ets_validation_benchmark(
        d1_train,
        d1_val,
        ["series_key"]
    )
)

display(d1_ets_validation)

d1_best_ets_config = ETS_CONFIGS[d1_best_ets_name]
d1_train_val = pd.concat([d1_train, d1_val], ignore_index=True)

d1_ets_test_preds, d1_ets_fit_time, d1_ets_inference_time, d1_ets_failures = run_ets(
    d1_train_val,
    d1_test,
    ["series_key"],
    d1_best_ets_config
)

print("Selected:", d1_best_ets_name)
print("Test sMAPE:", smape(d1_ets_test_preds["actual"], d1_ets_test_preds["forecast"]))
print("Test WAPE:", wape(d1_ets_test_preds["actual"], d1_ets_test_preds["forecast"]))
print("Test RMSE:", rmse(d1_ets_test_preds["actual"], d1_ets_test_preds["forecast"]))


## 7.2 Dataset 1 - LightGBM

Dataset 1 validation selected Config A.


In [ ]:
d1_lgbm_validation, d1_best_lgbm_name, d1_lgbm_val_preds = (
    lgbm_validation_benchmark(
        d1_train,
        d1_val,
        "series_key",
        ["store", "item"],
        LGBM_CONFIGS
    )
)

display(d1_lgbm_validation)

d1_best_lgbm_config = LGBM_CONFIGS[d1_best_lgbm_name]

d1_lgbm_model, d1_lgbm_features, d1_lgbm_fit_time = fit_lgbm(
    d1_train_val,
    "series_key",
    ["store", "item"],
    d1_best_lgbm_config
)

d1_lgbm_test_preds, d1_lgbm_inference_time = recursive_lgbm_forecast(
    d1_lgbm_model,
    d1_train_val,
    d1_test,
    "series_key",
    ["store", "item"],
    d1_lgbm_features
)

print("Selected:", d1_best_lgbm_name)
print("Test sMAPE:", smape(d1_lgbm_test_preds["actual"], d1_lgbm_test_preds["forecast"]))
print("Test WAPE:", wape(d1_lgbm_test_preds["actual"], d1_lgbm_test_preds["forecast"]))
print("Test RMSE:", rmse(d1_lgbm_test_preds["actual"], d1_lgbm_test_preds["forecast"]))


## 7.3 Dataset 1 - Chronos-2


In [ ]:
d1_chronos_test_preds, d1_chronos_inference_time = run_chronos(
    d1_train_val,
    d1_test,
    "series_key"
)

d1_chronos_coverage = 100 * np.mean(
    (d1_chronos_test_preds["actual"] >= d1_chronos_test_preds["lower_80"]) &
    (d1_chronos_test_preds["actual"] <= d1_chronos_test_preds["upper_80"])
)

d1_chronos_width = (
    d1_chronos_test_preds["upper_80"] -
    d1_chronos_test_preds["lower_80"]
).mean()

print("Test sMAPE:", smape(d1_chronos_test_preds["actual"], d1_chronos_test_preds["forecast"]))
print("Test WAPE:", wape(d1_chronos_test_preds["actual"], d1_chronos_test_preds["forecast"]))
print("Test RMSE:", rmse(d1_chronos_test_preds["actual"], d1_chronos_test_preds["forecast"]))
print("80% coverage:", d1_chronos_coverage)
print("Mean interval width:", d1_chronos_width)


# 8. Dataset 2 - M5

M5 is used as a **public industrial-style proxy**, not as private industrial data.

The original dataset contains 30,490 item-store series. To balance the comparison with Dataset 1, 50 series are sampled from each of the 10 stores using `random_state=42`, producing **500 series**.


In [ ]:
m5_sales = pd.read_csv(M5_SALES_PATH)
m5_calendar = pd.read_csv(M5_CALENDAR_PATH)
m5_prices = pd.read_csv(M5_PRICES_PATH)

m5_sample = (
    m5_sales
    .groupby("store_id", group_keys=False)
    .sample(n=50, random_state=42)
    .reset_index(drop=True)
)

id_cols = [
    "id", "item_id", "dept_id",
    "cat_id", "store_id", "state_id"
]

day_cols = [c for c in m5_sample.columns if c.startswith("d_")]

m5_long = m5_sample.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

m5_long = m5_long.merge(
    m5_calendar[["d", "date"]],
    on="d",
    how="left"
)

m5_long["date"] = pd.to_datetime(m5_long["date"])
m5_long = m5_long.sort_values(["id", "date"]).reset_index(drop=True)

m5_long["series_code"] = pd.factorize(m5_long["id"])[0]
m5_long["store_code"] = pd.factorize(m5_long["store_id"])[0]
m5_long["item_code"] = pd.factorize(m5_long["item_id"])[0]

m5_train, m5_val, m5_test = chronological_split(m5_long)

print("Sampled series:", m5_long["id"].nunique())
print("Rows:", len(m5_long))
print("Date range:", m5_long["date"].min(), "to", m5_long["date"].max())
print("Validation length per series:")
print(m5_val.groupby("id").size().value_counts())


## 8.1 M5 - ETS


In [ ]:
m5_ets_validation, m5_best_ets_name, m5_ets_val_preds = (
    ets_validation_benchmark(
        m5_train,
        m5_val,
        ["id"]
    )
)

display(m5_ets_validation)

m5_best_ets_config = ETS_CONFIGS[m5_best_ets_name]
m5_train_val = pd.concat([m5_train, m5_val], ignore_index=True)

m5_ets_test_preds, m5_ets_fit_time, m5_ets_inference_time, m5_ets_failures = run_ets(
    m5_train_val,
    m5_test,
    ["id"],
    m5_best_ets_config
)

print("Selected:", m5_best_ets_name)
print("Test sMAPE:", smape(m5_ets_test_preds["actual"], m5_ets_test_preds["forecast"]))
print("Test WAPE:", wape(m5_ets_test_preds["actual"], m5_ets_test_preds["forecast"]))
print("Test RMSE:", rmse(m5_ets_test_preds["actual"], m5_ets_test_preds["forecast"]))


## 8.2 M5 - LightGBM

M5 validation selected Config B.


In [ ]:
m5_lgbm_validation, m5_best_lgbm_name, m5_lgbm_val_preds = (
    lgbm_validation_benchmark(
        m5_train,
        m5_val,
        "id",
        ["series_code", "store_code", "item_code"],
        LGBM_CONFIGS
    )
)

display(m5_lgbm_validation)

m5_best_lgbm_config = LGBM_CONFIGS[m5_best_lgbm_name]

m5_lgbm_model, m5_lgbm_features, m5_lgbm_fit_time = fit_lgbm(
    m5_train_val,
    "id",
    ["series_code", "store_code", "item_code"],
    m5_best_lgbm_config
)

m5_lgbm_test_preds, m5_lgbm_inference_time = recursive_lgbm_forecast(
    m5_lgbm_model,
    m5_train_val,
    m5_test,
    "id",
    ["series_code", "store_code", "item_code"],
    m5_lgbm_features
)

print("Selected:", m5_best_lgbm_name)
print("Test sMAPE:", smape(m5_lgbm_test_preds["actual"], m5_lgbm_test_preds["forecast"]))
print("Test WAPE:", wape(m5_lgbm_test_preds["actual"], m5_lgbm_test_preds["forecast"]))
print("Test RMSE:", rmse(m5_lgbm_test_preds["actual"], m5_lgbm_test_preds["forecast"]))


## 8.3 M5 - Chronos-2


In [ ]:
m5_chronos_test_preds, m5_chronos_inference_time = run_chronos(
    m5_train_val,
    m5_test,
    "id"
)

m5_chronos_coverage = 100 * np.mean(
    (m5_chronos_test_preds["actual"] >= m5_chronos_test_preds["lower_80"]) &
    (m5_chronos_test_preds["actual"] <= m5_chronos_test_preds["upper_80"])
)

m5_chronos_width = (
    m5_chronos_test_preds["upper_80"] -
    m5_chronos_test_preds["lower_80"]
).mean()

print("Test sMAPE:", smape(m5_chronos_test_preds["actual"], m5_chronos_test_preds["forecast"]))
print("Test WAPE:", wape(m5_chronos_test_preds["actual"], m5_chronos_test_preds["forecast"]))
print("Test RMSE:", rmse(m5_chronos_test_preds["actual"], m5_chronos_test_preds["forecast"]))
print("80% coverage:", m5_chronos_coverage)
print("Mean interval width:", m5_chronos_width)


# 9. M5 external-covariate experiment

The selected M5 LightGBM configuration is held fixed while adding:

- sell price
- event indicator
- state-specific SNAP indicator

Future calendar/event/SNAP information and listed prices are treated as known covariates for the forecast horizon.


In [ ]:
calendar_covariates = m5_calendar[
    [
        "d", "wm_yr_wk",
        "event_name_1", "event_name_2",
        "snap_CA", "snap_TX", "snap_WI"
    ]
].copy()

m5_cov = m5_long.merge(
    calendar_covariates,
    on="d",
    how="left"
)

m5_cov = m5_cov.merge(
    m5_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

m5_cov["event_flag"] = (
    m5_cov["event_name_1"].notna() |
    m5_cov["event_name_2"].notna()
).astype(int)

m5_cov["snap"] = np.select(
    [
        m5_cov["state_id"] == "CA",
        m5_cov["state_id"] == "TX",
        m5_cov["state_id"] == "WI"
    ],
    [
        m5_cov["snap_CA"],
        m5_cov["snap_TX"],
        m5_cov["snap_WI"]
    ],
    default=0
)

# Matches the completed experiment.
m5_cov = m5_cov.sort_values(["id", "date"])
m5_cov["sell_price"] = (
    m5_cov.groupby("id")["sell_price"]
    .transform(lambda x: x.ffill().bfill())
)

m5_cov_train, m5_cov_val, m5_cov_test = chronological_split(m5_cov)
m5_cov_train_val = pd.concat(
    [m5_cov_train, m5_cov_val],
    ignore_index=True
)


In [ ]:
def make_covariate_features(df):
    data, base_features = make_lgbm_features(
        df,
        "id",
        ["series_code", "store_code", "item_code"]
    )

    covariates = ["sell_price", "event_flag", "snap"]
    return data, base_features + covariates


m5_cov_features, m5_cov_feature_cols = make_covariate_features(
    m5_cov_train_val
)
m5_cov_features = m5_cov_features.dropna(subset=m5_cov_feature_cols)

m5_cov_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    verbosity=-1,
    **m5_best_lgbm_config
)

start = time.perf_counter()
m5_cov_model.fit(
    m5_cov_features[m5_cov_feature_cols],
    m5_cov_features["sales"]
)
m5_cov_fit_time = time.perf_counter() - start


def recursive_covariate_forecast(model, history_df, future_df):
    history = {
        key: g.sort_values("date")["sales"].tolist()
        for key, g in history_df.groupby("id")
    }

    predictions = []
    start = time.perf_counter()

    for date in sorted(future_df["date"].unique()):
        date = pd.Timestamp(date)
        day = future_df[future_df["date"] == date]
        rows = []

        for _, r in day.iterrows():
            h = history[r["id"]]

            rows.append({
                "id": r["id"],
                "series_code": r["series_code"],
                "store_code": r["store_code"],
                "item_code": r["item_code"],
                "day_of_week": date.dayofweek,
                "month": date.month,
                "day_of_year": date.dayofyear,
                "lag_1": h[-1],
                "lag_7": h[-7],
                "lag_14": h[-14],
                "lag_28": h[-28],
                "rolling_mean_7": np.mean(h[-7:]),
                "rolling_mean_28": np.mean(h[-28:]),
                "sell_price": r["sell_price"],
                "event_flag": r["event_flag"],
                "snap": r["snap"]
            })

        X = pd.DataFrame(rows)[m5_cov_feature_cols]
        preds = np.maximum(model.predict(X), 0)

        for row, pred in zip(rows, preds):
            history[row["id"]].append(float(pred))
            predictions.append({
                "id": row["id"],
                "date": date,
                "forecast": float(pred)
            })

    inference_time = time.perf_counter() - start

    actual = future_df[
        ["id", "date", "sales"]
    ].rename(columns={"sales": "actual"})

    predictions = pd.DataFrame(predictions).merge(
        actual,
        on=["id", "date"],
        how="left"
    )

    return predictions, inference_time


m5_cov_test_preds, m5_cov_inference_time = recursive_covariate_forecast(
    m5_cov_model,
    m5_cov_train_val,
    m5_cov_test
)

print("Covariate sMAPE:", smape(m5_cov_test_preds["actual"], m5_cov_test_preds["forecast"]))
print("Covariate WAPE:", wape(m5_cov_test_preds["actual"], m5_cov_test_preds["forecast"]))
print("Covariate RMSE:", rmse(m5_cov_test_preds["actual"], m5_cov_test_preds["forecast"]))


# 10. Uncertainty calibration

ETS and LightGBM use a **validation-calibrated conformal residual interval**. Chronos-2 uses its native 10th/90th quantiles.

The important cleanup here is that the conformal calculation reuses the validation and test forecasts already generated above. It does **not** refit all ETS and LightGBM models a second time.


In [ ]:
def conformal_qhat(validation_predictions, alpha=0.20):
    scores = np.abs(
        validation_predictions["actual"].values -
        validation_predictions["forecast"].values
    )

    n = len(scores)
    q_level = min(
        1.0,
        np.ceil((n + 1) * (1 - alpha)) / n
    )

    return np.quantile(scores, q_level, method="higher")


def conformal_metrics(test_predictions, qhat):
    df = test_predictions.copy()

    df["lower_80"] = np.maximum(df["forecast"] - qhat, 0)
    df["upper_80"] = df["forecast"] + qhat

    coverage = 100 * np.mean(
        (df["actual"] >= df["lower_80"]) &
        (df["actual"] <= df["upper_80"])
    )

    width = (df["upper_80"] - df["lower_80"]).mean()
    return coverage, width


d1_ets_q = conformal_qhat(d1_ets_val_preds)
d1_lgbm_q = conformal_qhat(d1_lgbm_val_preds)
m5_ets_q = conformal_qhat(m5_ets_val_preds)
m5_lgbm_q = conformal_qhat(m5_lgbm_val_preds)

d1_ets_coverage, d1_ets_width = conformal_metrics(
    d1_ets_test_preds, d1_ets_q
)
d1_lgbm_coverage, d1_lgbm_width = conformal_metrics(
    d1_lgbm_test_preds, d1_lgbm_q
)
m5_ets_coverage, m5_ets_width = conformal_metrics(
    m5_ets_test_preds, m5_ets_q
)
m5_lgbm_coverage, m5_lgbm_width = conformal_metrics(
    m5_lgbm_test_preds, m5_lgbm_q
)

uncertainty_results = pd.DataFrame([
    ["Store-Item", "ETS", "Conformal residual", 80, d1_ets_coverage, d1_ets_width],
    ["Store-Item", "LightGBM", "Conformal residual", 80, d1_lgbm_coverage, d1_lgbm_width],
    ["Store-Item", "Chronos-2", "Native quantiles", 80, d1_chronos_coverage, d1_chronos_width],
    ["M5", "ETS", "Conformal residual", 80, m5_ets_coverage, m5_ets_width],
    ["M5", "LightGBM", "Conformal residual", 80, m5_lgbm_coverage, m5_lgbm_width],
    ["M5", "Chronos-2", "Native quantiles", 80, m5_chronos_coverage, m5_chronos_width]
], columns=[
    "Dataset",
    "Model",
    "Method",
    "Target Coverage (%)",
    "Observed Coverage (%)",
    "Mean Interval Width"
])

uncertainty_results[
    ["Observed Coverage (%)", "Mean Interval Width"]
] = uncertainty_results[
    ["Observed Coverage (%)", "Mean Interval Width"]
].round(2)

display(uncertainty_results)


# 11. Recorded final results

These are the measured results from the completed Google Colab runs used in the repository README. Runtime values are hardware-dependent.

## Point-forecast accuracy

| Dataset | Model | sMAPE | WAPE | RMSE |
|---|---|---:|---:|---:|
| Store-Item | ETS | 23.56% | 23.20% | 12.56 |
| Store-Item | LightGBM | 13.99% | 12.39% | 7.16 |
| Store-Item | Chronos-2 | **13.77%** | **12.07%** | **6.94** |
| M5 | ETS | **139.46%** | 75.59% | 2.28 |
| M5 | LightGBM | 141.22% | 78.85% | **2.06** |
| M5 | Chronos-2 | 162.30% | **67.90%** | 2.08 |

The very high M5 sMAPE values reflect the large number of zero and very low-demand observations, making WAPE and RMSE important supporting metrics.

## 80% uncertainty intervals

| Dataset | Model | Method | Observed coverage | Mean width |
|---|---|---|---:|---:|
| Store-Item | ETS | Conformal residual | 65.25% | 23.90 |
| Store-Item | LightGBM | Conformal residual | 85.20% | 20.07 |
| Store-Item | Chronos-2 | Native quantiles | **81.04%** | **17.54** |
| M5 | ETS | Conformal residual | **80.27%** | **1.98** |
| M5 | LightGBM | Conformal residual | 80.80% | 2.10 |
| M5 | Chronos-2 | Native quantiles | 78.64% | 2.65 |

Interval widths should only be compared within a dataset because Store-Item and M5 operate at different sales scales.

## Computational cost

| Dataset | Model | Fit / task-specific training | Inference |
|---|---|---:|---:|
| Store-Item | ETS | 262.80 s | 21.46 s |
| Store-Item | LightGBM | 21.91 s | **0.60 s** |
| Store-Item | Chronos-2 | 0 s local fitting | 7.15 s |
| M5 | ETS | 215.50 s | 20.44 s |
| M5 | LightGBM | 15.64 s | **0.50 s** |
| M5 | Chronos-2 | 0 s local fitting | 5.99 s |

Chronos-2's 0 s value means **no task-specific fitting**, not that the pretrained model required no training.

## M5 covariate experiment

| LightGBM variant | sMAPE | WAPE | RMSE |
|---|---:|---:|---:|
| Historical features | 141.22% | **78.85%** | **2.06** |
| + Price, event, SNAP | **141.03%** | 80.95% | 2.11 |

The external covariates did not provide a consistent overall improvement.


# 12. Conclusions

1. **Model rankings are dataset-dependent.** Chronos-2 wins all three point metrics on Store-Item, while on M5 ETS wins sMAPE, Chronos-2 wins WAPE, and LightGBM wins RMSE.
2. **Foundation models are competitive but not universally superior.** Chronos-2 is strongest on the simpler benchmark, but its advantage disappears on M5.
3. **LightGBM provides a strong accuracy-cost trade-off.** It is close to Chronos-2 on Store-Item, achieves the best M5 RMSE, and has the fastest inference time on both datasets.
4. **Classical statistical methods remain competitive.** ETS performs poorly on Store-Item but achieves the best M5 sMAPE and strong M5 uncertainty calibration.
5. **Uncertainty behavior is dataset-dependent.** Chronos-2 is strongest on Store-Item, while ETS and LightGBM are especially well calibrated on M5.
6. **More covariates do not automatically improve forecasts.** Price, event, and SNAP features did not consistently improve M5 LightGBM performance.

## Bottom line

There is no universally best forecasting approach across datasets, accuracy metrics, uncertainty requirements, and computational constraints. Industrial forecasting benchmarks should evaluate model families under the characteristics and deployment constraints of the target problem rather than relying on a single public leaderboard.
